# Iowa Housing — Features & Model

**Person 2 notebook.**  
This notebook builds, evaluates, and exports the machine-learning model for the Iowa House-Prices Kaggle dataset.  
Target variable: `SalePrice`. We use exactly 7 hand-picked features throughout.

## Step 1 — Load Data & Select Features

Load `data/train.csv`, define the 7 required features, build `X` (feature matrix) and `y` (target vector), then confirm shapes and column names.

In [1]:
import pandas as pd

# Constants
FEATURES = [
    'LotArea',
    'YearBuilt',
    '1stFlrSF',
    '2ndFlrSF',
    'FullBath',
    'BedroomAbvGr',
    'TotRmsAbvGrd',
]
TARGET = 'SalePrice'

# Load
df = pd.read_csv('data/train.csv')

# Build X and y
X = df[FEATURES]
y = df[TARGET]

# Confirm
print('Dataset shape  :', df.shape)
print('X shape        :', X.shape)
print('y shape        :', y.shape)
print()
print('Features present:', list(X.columns))
print()
print('All 7 features in X?', all(f in X.columns for f in FEATURES))
print()
X.head()

Dataset shape  : (1460, 81)
X shape        : (1460, 7)
y shape        : (1460,)

Features present: ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']

All 7 features in X? True



,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
0,8450,2003,856,854,2,3,8
1,9600,1976,1262,0,2,3,6
2,11250,2001,920,866,2,3,6
3,9550,1915,961,756,1,3,7
4,14260,2000,1145,1053,2,4,9


## Step 2 — Justify the Features

For each of the 7 features we print its Pearson correlation with `SalePrice` and give a one-line rationale to use in the presentation.

In [2]:
# Correlations with SalePrice
correlations = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)

reasons = {
    'LotArea'     : 'Larger lots command a premium — more land = more value.',
    'YearBuilt'   : 'Newer homes are in better condition and built to modern standards.',
    '1stFlrSF'    : 'First-floor square footage is the primary driver of interior space.',
    '2ndFlrSF'    : 'Additional floor area directly increases livable space and price.',
    'FullBath'    : 'Full bathrooms are a key buyer priority — strong proxy for quality.',
    'BedroomAbvGr': 'Bedroom count is one of the first filters buyers use when searching.',
    'TotRmsAbvGrd': 'Total rooms capture overall house size beyond just bedrooms.',
}

print(f'{"Feature":<15} {"Correlation":>12}   Reason')
print('-' * 80)
for feat, corr in correlations.items():
    print(f'{feat:<15} {corr:>12.4f}   {reasons[feat]}')


Feature          Correlation   Reason
--------------------------------------------------------------------------------
LotArea               0.2638   Larger lots command a premium — more land = more value.
YearBuilt             0.5229   Newer homes are in better condition and built to modern standards.
1stFlrSF              0.6059   First-floor square footage is the primary driver of interior space.
2ndFlrSF              0.3193   Additional floor area directly increases livable space and price.
FullBath              0.5607   Full bathrooms are a key buyer priority — strong proxy for quality.
BedroomAbvGr          0.1682   Bedroom count is one of the first filters buyers use when searching.
TotRmsAbvGrd          0.5337   Total rooms capture overall house size beyond just bedrooms.


## Step 3 — Handle Missing Values

Check `X.isna().sum()` for each of the 7 features. If any column has nulls, fill them with the column median. Show the before and after counts.

In [3]:
# Before: count nulls per feature
missing_before = X.isna().sum()
print('=== Missing values BEFORE fill ===')
print(missing_before)
print()

# Fill nulls with column median (only touches columns that need it)
X = X.fillna(X.median())

# After: confirm zeros
missing_after = X.isna().sum()
print('=== Missing values AFTER fill ===')
print(missing_after)
print()
print('Any nulls remaining?', X.isna().any().any())


=== Missing values BEFORE fill ===
LotArea         0
YearBuilt       0
1stFlrSF        0
2ndFlrSF        0
FullBath        0
BedroomAbvGr    0
TotRmsAbvGrd    0
dtype: int64

=== Missing values AFTER fill ===
LotArea         0
YearBuilt       0
1stFlrSF        0
2ndFlrSF        0
FullBath        0
BedroomAbvGr    0
TotRmsAbvGrd    0
dtype: int64

Any nulls remaining? False


## Step 4 — Train / Test Split

Split `X` and `y` into 80 % training and 20 % test sets using `random_state=1` for reproducibility. Print all four resulting shapes.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test  shape:', y_test.shape)
print()
print('Train rows:', len(X_train), '| Test rows:', len(X_test))
print('Split ratio: {:.0f}/{:.0f}'.format(
    100 * len(X_train) / len(X),
    100 * len(X_test)  / len(X)
))


X_train shape: (1168, 7)
X_test  shape: (292, 7)
y_train shape: (1168,)
y_test  shape: (292,)

Train rows: 1168 | Test rows: 292
Split ratio: 80/20


## Step 5 — Baseline Model

A naive baseline that predicts the **median `SalePrice`** of the training set for every house. Its MAE on the test set is the benchmark every real model must beat.

In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# Baseline: always predict the training-set median
median_price = y_train.median()
baseline_preds = np.full(shape=len(y_test), fill_value=median_price)

baseline_mae = mean_absolute_error(y_test, baseline_preds)

print('Training-set median SalePrice : ${:,.0f}'.format(median_price))
print('Baseline MAE (test set)       : ${:,.0f}'.format(baseline_mae))
print()
print('=> Any real model must achieve MAE < ${:,.0f} to beat the baseline.'.format(baseline_mae))


Training-set median SalePrice : $164,945
Baseline MAE (test set)       : $56,621

=> Any real model must achieve MAE < $56,621 to beat the baseline.


## Step 6 — Train Two Models

Train a **Decision Tree** and a **Random Forest** on the training set using the exact hyperparameters specified. Confirm both fit successfully.

In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Decision Tree
dt_model = DecisionTreeRegressor(
    max_depth=8,
    min_samples_leaf=5,
    random_state=1
)
dt_model.fit(X_train, y_train)

# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=1,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

print('Decision Tree  -> fitted:', hasattr(dt_model, 'tree_'))
print('  max_depth=8, min_samples_leaf=5, random_state=1')
print()
print('Random Forest  -> fitted:', hasattr(rf_model, 'estimators_'))
print('  n_estimators=300, min_samples_leaf=2, random_state=1, n_jobs=-1')
print()
print('Both models trained successfully!')


Decision Tree  -> fitted: True
  max_depth=8, min_samples_leaf=5, random_state=1

Random Forest  -> fitted: True
  n_estimators=300, min_samples_leaf=2, random_state=1, n_jobs=-1

Both models trained successfully!


## Step 7 — Evaluate & Compare

Compute **MAE**, **RMSE**, and **R²** on the test set for all three models (Baseline, Decision Tree, Random Forest) and display a clean comparison table.

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2   = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = [
    evaluate('Baseline',      y_test, baseline_preds),
    evaluate('Decision Tree', y_test, dt_model.predict(X_test)),
    evaluate('Random Forest', y_test, rf_model.predict(X_test)),
]

import pandas as pd
results_df = pd.DataFrame(results).set_index('Model')

# Pretty-print
print('{:<15} {:>12} {:>12} {:>8}'.format('Model', 'MAE ($)', 'RMSE ($)', 'R2'))
print('-' * 52)
for idx, row in results_df.iterrows():
    print('{:<15} {:>12,.0f} {:>12,.0f} {:>8.4f}'.format(
        idx, row['MAE'], row['RMSE'], row['R2']))

print()
results_df


Model                MAE ($)     RMSE ($)       R2
----------------------------------------------------
Baseline              56,621       85,142  -0.0164
Decision Tree         26,968       40,787   0.7667
Random Forest         22,394       33,526   0.8424



,MAE,RMSE,R2
Model,,,
Baseline,56621.147260,85142.119894,-0.016438
Decision Tree,26968.238223,40786.705916,0.766746
Random Forest,22394.175551,33525.947826,0.842401


## Step 8 — Pick the Final Model

Select the model with the **lowest MAE** on the test set. We don’t assume a winner — we let the numbers decide.

In [8]:
# Map model names to (model object, MAE)
candidates = {
    'Decision Tree': (dt_model, results_df.loc['Decision Tree', 'MAE']),
    'Random Forest': (rf_model, results_df.loc['Random Forest', 'MAE']),
}

# Pick winner by lowest MAE
winner_name = min(candidates, key=lambda k: candidates[k][1])
best_model, winner_mae = candidates[winner_name]

print('=== Model Comparison (MAE on test set) ===')
for name, (_, mae) in candidates.items():
    marker = '  <-- WINNER' if name == winner_name else ''
    print('  {:<15}: ${:,.0f}{}'.format(name, mae, marker))

print()
print('Final model selected : {}'.format(winner_name))
print('Test MAE             : ${:,.0f}'.format(winner_mae))
print('Test RMSE            : ${:,.0f}'.format(results_df.loc[winner_name, 'RMSE']))
print('Test R2              : {:.4f}'.format(results_df.loc[winner_name, 'R2']))
print()
print('Reason: {} achieves the lowest MAE, meaning on average '
       'its predictions are ${:,.0f} away from the true sale price — '
       '${:,.0f} better than the baseline.'.format(
    winner_name,
    winner_mae,
    results_df.loc['Baseline', 'MAE'] - winner_mae
))


=== Model Comparison (MAE on test set) ===
  Decision Tree  : $26,968
  Random Forest  : $22,394  <-- WINNER

Final model selected : Random Forest
Test MAE             : $22,394
Test RMSE            : $33,526
Test R2              : 0.8424

Reason: Random Forest achieves the lowest MAE, meaning on average its predictions are $22,394 away from the true sale price — $34,227 better than the baseline.


## Step 9 — Export Artifacts

Save the winning model and the features list to the repo root using `joblib.dump`. These two files are the primary deliverables to Person 3.

In [9]:
import joblib
import os

# Export the best model
joblib.dump(best_model, 'iowa_model.pkl')

# Export the feature list
joblib.dump(FEATURES, 'iowa_features.pkl')

# Confirm files exist and show sizes
for fname in ['iowa_model.pkl', 'iowa_features.pkl']:
    size_kb = os.path.getsize(fname) / 1024
    print('Saved: {:20s}  ({:.1f} KB)'.format(fname, size_kb))

print()
print('Both artifacts exported successfully.')


Saved: iowa_model.pkl        (13613.7 KB)
Saved: iowa_features.pkl     (0.1 KB)

Both artifacts exported successfully.


## Step 10 — Verify the Artifacts

Reload `iowa_model.pkl` and `iowa_features.pkl` fresh from disk (simulating what Person 3 will do), then run a test prediction on the example house to confirm the round-trip works end-to-end.

In [10]:
import joblib
import pandas as pd

# Reload both artifacts from disk
loaded_model    = joblib.load('iowa_model.pkl')
loaded_features = joblib.load('iowa_features.pkl')

print('Loaded model type :', type(loaded_model).__name__)
print('Loaded features   :', loaded_features)
print()

# Example house from the spec
example = {
    'LotArea'     : 8450,
    'YearBuilt'   : 2003,
    '1stFlrSF'   : 856,
    '2ndFlrSF'   : 854,
    'FullBath'    : 2,
    'BedroomAbvGr': 3,
    'TotRmsAbvGrd': 8,
}

# Build a single-row DataFrame in the exact feature order
example_df = pd.DataFrame([example])[loaded_features]

predicted_price = loaded_model.predict(example_df)[0]
print('Example house input:')
print(example)
print()
print('Predicted SalePrice: ${:,.0f}'.format(predicted_price))


Loaded model type : RandomForestRegressor
Loaded features   : ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']



Example house input:
{'LotArea': 8450, 'YearBuilt': 2003, '1stFlrSF': 856, '2ndFlrSF': 854, 'FullBath': 2, 'BedroomAbvGr': 3, 'TotRmsAbvGrd': 8}

Predicted SalePrice: $206,060


## Step 11 — Create `sample_houses.csv`

Build a small CSV of realistic example houses whose column names match the 7 features **exactly**. This file is used by Person 3 to test the Streamlit demo without having to enter values manually.

In [11]:
import pandas as pd

sample_houses = pd.DataFrame([
    # Small/older starter home
    {'LotArea': 5500,  'YearBuilt': 1965, '1stFlrSF': 700,  '2ndFlrSF': 0,
     'FullBath': 1, 'BedroomAbvGr': 2, 'TotRmsAbvGrd': 5},
    # Average mid-range home (the spec example house)
    {'LotArea': 8450,  'YearBuilt': 2003, '1stFlrSF': 856,  '2ndFlrSF': 854,
     'FullBath': 2, 'BedroomAbvGr': 3, 'TotRmsAbvGrd': 8},
    # Spacious newer family home
    {'LotArea': 12000, 'YearBuilt': 2010, '1stFlrSF': 1200, '2ndFlrSF': 900,
     'FullBath': 3, 'BedroomAbvGr': 4, 'TotRmsAbvGrd': 10},
    # Large luxury home
    {'LotArea': 20000, 'YearBuilt': 2015, '1stFlrSF': 2000, '2ndFlrSF': 1500,
     'FullBath': 4, 'BedroomAbvGr': 5, 'TotRmsAbvGrd': 12},
    # Compact bungalow
    {'LotArea': 4500,  'YearBuilt': 1950, '1stFlrSF': 850,  '2ndFlrSF': 0,
     'FullBath': 1, 'BedroomAbvGr': 2, 'TotRmsAbvGrd': 4},
])

# Ensure column order matches FEATURES exactly
sample_houses = sample_houses[FEATURES]

sample_houses.to_csv('sample_houses.csv', index=False)

print('sample_houses.csv saved with', len(sample_houses), 'rows')
print('Columns:', list(sample_houses.columns))
print()
print(sample_houses.to_string(index=False))


sample_houses.csv saved with 5 rows
Columns: ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']

 LotArea  YearBuilt  1stFlrSF  2ndFlrSF  FullBath  BedroomAbvGr  TotRmsAbvGrd
    5500       1965       700         0         1             2             5
    8450       2003       856       854         2             3             8
   12000       2010      1200       900         3             4            10
   20000       2015      2000      1500         4             5            12
    4500       1950       850         0         1             2             4


## Step 12 — Final Check

Confirm all three required deliverables exist in the repo root and print a complete metrics summary ready to report.

In [12]:
import os

deliverables = ['iowa_model.pkl', 'iowa_features.pkl', 'sample_houses.csv']

print('=' * 55)
print('DELIVERABLE CHECK')
print('=' * 55)
all_ok = True
for fname in deliverables:
    exists = os.path.isfile(fname)
    size_kb = os.path.getsize(fname) / 1024 if exists else 0
    status = 'OK' if exists else 'MISSING'
    print('  [{:^7}]  {:25s} ({:.1f} KB)'.format(status, fname, size_kb))
    if not exists:
        all_ok = False
print()
print('All deliverables present:', all_ok)

print()
print('=' * 55)
print('MODEL METRICS SUMMARY (for presentation)')
print('=' * 55)
print('Final model  : {}'.format(winner_name))
print('Features     : {} features'.format(len(FEATURES)))
print('Training rows: {}'.format(len(X_train)))
print('Test rows    : {}'.format(len(X_test)))
print()
print('{:<12} {:>10} {:>10} {:>8}'.format('Model', 'MAE', 'RMSE', 'R2'))
print('-' * 44)
for idx, row in results_df.iterrows():
    print('{:<12} {:>10,.0f} {:>10,.0f} {:>8.4f}'.format(
        idx, row['MAE'], row['RMSE'], row['R2']))
print()
print('Improvement over baseline:')
print('  MAE  reduced by ${:,.0f} ({:.0f}%)'.format(
    results_df.loc['Baseline','MAE'] - winner_mae,
    100*(results_df.loc['Baseline','MAE'] - winner_mae)/results_df.loc['Baseline','MAE']
))
print('  RMSE reduced by ${:,.0f}'.format(
    results_df.loc['Baseline','RMSE'] - results_df.loc[winner_name,'RMSE']
))
print('  R2 went from {:.4f} to {:.4f}'.format(
    results_df.loc['Baseline','R2'], results_df.loc[winner_name,'R2']
))


DELIVERABLE CHECK
  [  OK   ]  iowa_model.pkl            (13613.7 KB)
  [  OK   ]  iowa_features.pkl         (0.1 KB)
  [  OK   ]  sample_houses.csv         (0.2 KB)

All deliverables present: True

MODEL METRICS SUMMARY (for presentation)
Final model  : Random Forest
Features     : 7 features
Training rows: 1168
Test rows    : 292

Model               MAE       RMSE       R2
--------------------------------------------
Baseline         56,621     85,142  -0.0164
Decision Tree     26,968     40,787   0.7667
Random Forest     22,394     33,526   0.8424

Improvement over baseline:
  MAE  reduced by $34,227 (60%)
  RMSE reduced by $51,616
  R2 went from -0.0164 to 0.8424
